*0.3 Classical NLP*

# Word2Vec

**The situation.** Before embedding APIs (0.2), how did a machine know "refund" and "reimbursement" were related? A search team in 2015 had no OpenAI; they had a lot of text. Word2Vec is the answer they used, and the idea underneath every modern embedding: *a word is known by the company it keeps*.

**Word2Vec.** Slide a window over the text. Train a small network to predict a word from its neighbours (or the neighbours from the word). The network's hidden layer, one vector per word, becomes the embedding. Words used in the same contexts end with similar vectors. Trained with gensim on your own corpus in minutes.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Train on real text**: 20 Newsgroups posts (about 3,000 posts across five topics), a few minutes on CPU. Then ask the model what is near "engine".

In [2]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.datasets import fetch_20newsgroups

topics = ["rec.autos", "rec.motorcycles", "sci.med", "sci.space", "comp.graphics"]
posts = fetch_20newsgroups(
    subset="train", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)
sentences = []
for post in posts.data:
    tokens = simple_preprocess(post)  # lowercase, split, drop punctuation
    if len(tokens) > 5:
        sentences.append(tokens)

model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4, epochs=20, seed=0)
print("vocabulary:", len(model.wv), "words | vector size:", model.wv.vector_size)
for word in ("engine", "doctor", "orbit"):
    neighbours = []
    for neighbour, score in model.wv.most_similar(word, topn=4):
        neighbours.append(f"{neighbour} ({score:.2f})")
    print(f"{word:<8} → " + ", ".join(neighbours))
assert model.wv.similarity("engine", "car") > model.wv.similarity("engine", "doctor")

vocabulary: 7792 words | vector size: 100
engine   → torque (0.75), tank (0.74), suspension (0.74), oil (0.73)
doctor   → she (0.76), lady (0.73), judge (0.71), wife (0.70)
orbit    → comet (0.81), earth (0.79), leo (0.79), spacecraft (0.76)


**Reading the output.** "engine" sits near car and bike words, "doctor" near medical words, "orbit" near space words — learned from nothing but which words appear near which. The similarity check at the end is the same dot-product-of-unit-vectors from 0.2.

```
"... the car's engine stalled ..."      window: [the, car's, ___, stalled]
"... the bike's engine roared ..."      window: [the, bike's, ___, roared]
                                        → "engine" predicted from similar neighbours
                                        → "car's" and "bike's" get similar vectors too
```

**The rule to remember.** Word2Vec learns word vectors from co-occurrence, from any text you have. It is the ancestor of every embedding model, and still useful when your vocabulary is private (product codes, internal jargon) and an API model has never seen it.

| Use it when | Don't when | Instead use |
|---|---|---|
| domain-specific vocabulary, offline constraints, a fast on-prem baseline | you need sentence meaning, not word meaning; general text | sentence embeddings (OpenAI, sentence-transformers) |

**Watch out**
- One vector per word, regardless of context: "charge" (money) and "charge" (battery) share one vector.
- Words below `min_count` are dropped and unknown at query time — the problem FastText (item 18) solves.
- Results depend on corpus size: a few thousand posts give rough neighbours; millions give good ones.